# 24.5 Multi-objective optimization (NSGA-II)

NSGA-II searches for a diverse Pareto front instead of pretending there is only one objective.

Conflicting goals are ranked by dominance, not by a premature weighted sum. NSGA-II keeps non-dominated fronts and then uses crowding distance so endpoints and spread survive selection.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 245
rng = np.random.default_rng(SEED)


def dominates(a, b):
    return bool(np.all(a <= b) and np.any(a < b))


def non_dominated_sort(objectives):
    objectives = np.asarray(objectives, dtype=float)
    count = objectives.shape[0]
    dominates_list = [[] for _ in range(count)]
    domination_count = np.zeros(count, dtype=int)
    fronts = [[]]
    for p in range(count):
        for q in range(count):
            if dominates(objectives[p], objectives[q]):
                dominates_list[p].append(q)
            elif dominates(objectives[q], objectives[p]):
                domination_count[p] += 1
        if domination_count[p] == 0:
            fronts[0].append(p)
    index = 0
    while len(fronts[index]) > 0:
        next_front = []
        for p in fronts[index]:
            for q in dominates_list[p]:
                domination_count[q] -= 1
                if domination_count[q] == 0:
                    next_front.append(q)
        index += 1
        fronts.append(next_front)
    return fronts[:-1]


def crowding_distance(objectives, front):
    objectives = np.asarray(objectives, dtype=float)
    distance = np.zeros(len(front), dtype=float)
    if len(front) == 0:
        return distance
    front_objectives = objectives[np.array(front)]
    objective_count = front_objectives.shape[1]
    for objective_index in range(objective_count):
        order = np.argsort(front_objectives[:, objective_index])
        distance[order[0]] = np.inf
        distance[order[-1]] = np.inf
        span = front_objectives[order[-1], objective_index] - front_objectives[order[0], objective_index]
        if span == 0.0:
            continue
        for rank in range(1, len(front) - 1):
            previous_value = front_objectives[order[rank - 1], objective_index]
            next_value = front_objectives[order[rank + 1], objective_index]
            distance[order[rank]] += (next_value - previous_value) / span
    return distance


def hypervolume_2d(objectives, reference):
    front = non_dominated_sort(objectives)[0]
    points = objectives[np.array(front)]
    points = points[np.argsort(points[:, 0])]
    volume = 0.0
    previous_f2 = reference[1]
    for point in points:
        width = max(0.0, reference[0] - point[0])
        height = max(0.0, previous_f2 - point[1])
        volume += width * height
        previous_f2 = min(previous_f2, point[1])
    return float(volume)


def objective_d1(population):
    x = population[:, 0]
    return np.column_stack([np.square(x), np.square(x - 2.0)])


def objective_d2(population):
    return np.column_stack([np.sum(np.square(population - np.array([0.0, 0.0])), axis=1), np.sum(np.square(population - np.array([2.0, 2.0])), axis=1)])


def objective_d3(population):
    ras = 20.0 + np.sum(np.square(population) - 10.0 * np.cos(2.0 * np.pi * population), axis=1)
    ack = -20.0 * np.exp(-0.2 * np.sqrt(np.mean(np.square(population - 1.0), axis=1))) - np.exp(np.mean(np.cos(2.0 * np.pi * (population - 1.0)), axis=1)) + 20.0 + np.e
    return np.column_stack([ras, ack])


def objective_d4(population):
    base = objective_d2(population)
    violation = np.maximum(0.0, population[:, 0] + population[:, 1] - 1.5)
    penalty = 30.0 * np.square(violation)
    return base + penalty[:, None]


def objective_d5(population):
    accuracy_loss = np.sum(np.square(population[:, :5] - 0.5), axis=1)
    latency = np.sum(np.abs(population[:, 5:10]), axis=1) + 0.2 * np.sum(np.square(population[:, :5]), axis=1)
    instability = np.sum(np.square(population[:, 10:] + 0.25), axis=1)
    return np.column_stack([accuracy_loss, latency, instability])


def make_nsga_ladder():
    return [
        {"name": "D1 1-D two-objective toy", "dim": 1, "bounds": (-1.0, 3.0), "objective": objective_d1, "reference": np.array([10.0, 10.0])},
        {"name": "D2 2-D sphere tradeoff", "dim": 2, "bounds": (-1.0, 3.0), "objective": objective_d2, "reference": np.array([20.0, 20.0])},
        {"name": "D3 multimodal bi-objective", "dim": 2, "bounds": (-4.0, 4.0), "objective": objective_d3, "reference": np.array([80.0, 25.0])},
        {"name": "D4 constrained bi-objective", "dim": 2, "bounds": (-1.0, 3.0), "objective": objective_d4, "reference": np.array([50.0, 50.0])},
        {"name": "D5 12-D three-objective tradeoff", "dim": 12, "bounds": (-2.0, 2.0), "objective": objective_d5, "reference": np.array([35.0, 35.0, 35.0])},
    ]


def approximate_hypervolume(objectives, reference, samples=20000, rng=None):
    objectives = np.asarray(objectives, dtype=float)
    if objectives.shape[1] == 2:
        return hypervolume_2d(objectives, reference)
    if rng is None:
        rng = np.random.default_rng(SEED)
    lower = np.min(objectives, axis=0)
    random_points = rng.uniform(lower, reference, size=(samples, objectives.shape[1]))
    dominated = np.zeros(samples, dtype=bool)
    for point in objectives:
        dominated = dominated | np.all(point <= random_points, axis=1)
    box = np.prod(reference - lower)
    return float(box * np.mean(dominated))


def binary_tournament(population, objectives, fronts, rng):
    ranks = np.empty(population.shape[0], dtype=int)
    crowding = np.zeros(population.shape[0], dtype=float)
    for rank, front in enumerate(fronts):
        ranks[np.array(front)] = rank
        distances = crowding_distance(objectives, front)
        crowding[np.array(front)] = distances
    a, b = rng.choice(population.shape[0], size=2, replace=False)
    if ranks[a] < ranks[b]:
        return population[a]
    if ranks[b] < ranks[a]:
        return population[b]
    if crowding[a] >= crowding[b]:
        return population[a]
    return population[b]


def make_children(population, objectives, fronts, bounds, rng):
    lower, upper = bounds
    children = []
    while len(children) < population.shape[0]:
        parent_a = binary_tournament(population, objectives, fronts, rng)
        parent_b = binary_tournament(population, objectives, fronts, rng)
        alpha = rng.uniform(0.25, 0.75, size=population.shape[1])
        child = alpha * parent_a + (1.0 - alpha) * parent_b
        child = child + rng.normal(0.0, 0.12, size=child.shape)
        child = np.clip(child, lower, upper)
        children.append(child)
    return np.array(children)


def environmental_select(combined, combined_objectives, population_size, use_crowding=True):
    fronts = non_dominated_sort(combined_objectives)
    selected = []
    for front in fronts:
        if len(selected) + len(front) <= population_size:
            selected.extend(front)
        else:
            remaining = population_size - len(selected)
            if use_crowding:
                distances = crowding_distance(combined_objectives, front)
                order = np.argsort(distances)[::-1]
            else:
                order = np.arange(len(front))
            selected.extend([front[int(index)] for index in order[:remaining]])
            break
    selected = np.array(selected, dtype=int)
    return combined[selected]


def nsga2_optimize(objective, dim, bounds, reference, population_size=70, generations=55, use_crowding=True, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    lower, upper = bounds
    population = rng.uniform(lower, upper, size=(population_size, dim))
    history = []
    for generation in range(generations):
        objectives = objective(population)
        fronts = non_dominated_sort(objectives)
        children = make_children(population, objectives, fronts, bounds, rng)
        combined = np.vstack([population, children])
        combined_objectives = objective(combined)
        population = environmental_select(combined, combined_objectives, population_size, use_crowding=use_crowding)
        current_objectives = objective(population)
        history.append(approximate_hypervolume(current_objectives, reference, samples=6000, rng=rng))
    objectives = objective(population)
    return {"population": population, "objectives": objectives, "fronts": non_dominated_sort(objectives), "hypervolume": approximate_hypervolume(objectives, reference, samples=12000, rng=rng), "history": np.array(history)}


## The concept, built once (D1): non-dominated sorting

For minimization, $a$ dominates $b$ when

$$f_k(a)\le f_k(b)\;\text{for all }k,\qquad f_j(a)\lt f_j(b)\;\text{for at least one }j.$$

The lesson's six pairs have a first front with exactly four points: $(1,5),(2,3),(3,2),(5,1)$.

In [ ]:

lesson_pairs = np.array([[1.0, 5.0], [2.0, 3.0], [3.0, 2.0], [5.0, 1.0], [3.0, 4.0], [4.0, 3.0]])
fronts = non_dominated_sort(lesson_pairs)
front_points = lesson_pairs[np.array(fronts[0])]

print("first front:", front_points)

assert len(fronts[0]) == 4
assert any(np.all(point == np.array([2.0, 3.0])) for point in front_points)
assert dominates(np.array([2.0, 3.0]), np.array([3.0, 4.0]))


## Crowding distance protects spread

Boundary points get infinite distance. For $(2,3)$ inside the front, the lesson contribution is $(3-1)/(5-1)=0.50$ for objective 1 and $(5-2)/(5-1)=0.75$ for objective 2, so the crowding distance is $1.25$.

In [ ]:

front = fronts[0]
distances = crowding_distance(lesson_pairs, front)
front_list = list(front)
point_index = int(np.where(np.all(lesson_pairs == np.array([2.0, 3.0]), axis=1))[0][0])
local_index = front_list.index(point_index)
lesson_distance = distances[local_index]

print("crowding distances:", distances)
print("distance for (2,3):", lesson_distance)

assert np.isinf(np.max(distances))
assert np.isclose(lesson_distance, 1.25)


## The dataset ladder

The inline NSGA-II ladder moves from a known 1-D Pareto toy to 2-D sphere tradeoffs, multimodal objectives, constrained feasibility penalties, and a 12-D three-objective accuracy-latency-stability tradeoff.

In [ ]:

ladder = make_nsga_ladder()

for rung in ladder:
    preview_rng = np.random.default_rng(SEED + rung["dim"])
    lower, upper = rung["bounds"]
    sample = preview_rng.uniform(lower, upper, size=(5, rung["dim"]))
    values = rung["objective"](sample)
    print(rung["name"], "population shape", sample.shape, "objective shape", values.shape, "sample", np.round(values[:2], 3))


In [ ]:

results = []

for index, rung in enumerate(ladder):
    result = nsga2_optimize(rung["objective"], rung["dim"], rung["bounds"], rung["reference"], population_size=70, generations=55, use_crowding=True, rng=np.random.default_rng(SEED + index))
    results.append(result)
    print(f"{rung['name']}: hypervolume={result['hypervolume']:.4f}, front0={len(result['fronts'][0])}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))

for index, rung in enumerate(ladder):
    ax = axes[0, index]
    objectives = results[index]["objectives"]
    front0 = results[index]["fronts"][0]
    if objectives.shape[1] == 2:
        dominated_indices = [row for row in range(objectives.shape[0]) if row not in front0]
        ax.scatter(objectives[dominated_indices, 0], objectives[dominated_indices, 1], c="lightgray", s=15, label="dominated")
        ax.scatter(objectives[front0, 0], objectives[front0, 1], c="red", s=25, label="front")
        ax.set_xlabel("objective 1")
        ax.set_ylabel("objective 2")
    else:
        ax.scatter(objectives[:, 0], objectives[:, 1], c=objectives[:, 2], cmap="viridis", s=18)
        ax.scatter(objectives[front0, 0], objectives[front0, 1], facecolors="none", edgecolors="red", s=45)
        ax.set_xlabel("accuracy loss")
        ax.set_ylabel("latency")
    ax.set_title(rung["name"].split()[0])

for index, result in enumerate(results):
    axes[1, index].plot(result["history"])
    axes[1, index].set_title("hypervolume")
    axes[1, index].set_xlabel("generation")

fig.suptitle("Pareto-front panels and hypervolume curves")
plt.tight_layout()


## Pitfall on D5: keeping only rank

Rank-only selection can crowd the first front into a small region. The fix uses normalized crowding distance and endpoint preservation when a front overflows the survivor budget.

In [ ]:

d5 = ladder[-1]

rank_only = nsga2_optimize(d5["objective"], d5["dim"], d5["bounds"], d5["reference"], population_size=70, generations=55, use_crowding=False, rng=np.random.default_rng(999))
crowded = nsga2_optimize(d5["objective"], d5["dim"], d5["bounds"], d5["reference"], population_size=70, generations=55, use_crowding=True, rng=np.random.default_rng(999))

rank_spread = np.mean(np.std(rank_only["objectives"][rank_only["fronts"][0]], axis=0))
crowd_spread = np.mean(np.std(crowded["objectives"][crowded["fronts"][0]], axis=0))

print("rank-only hypervolume", rank_only["hypervolume"], "front spread", rank_spread)
print("crowding hypervolume", crowded["hypervolume"], "front spread", crowd_spread)


## Evaluate it + Practice

- Compare the reported hypervolume against a seeded random-search baseline with the same evaluation budget.
- Sanity check: D1 must hit the lesson's worked calculation before trusting D2–D5.
- Ablation: disable crowding distance during environmental selection should make the hardest rung worse or less stable.
- Failure signals: population variance collapses too early, bounds are violated, or the summary curve improves only by one lucky sample.
- Re-run with another seed only after the structural checks pass; do not tune from a single trace.

Practice prompts:
1. Change the hardest rung budget and explain whether convergence improves per objective call.
2. Add one diagnostic for diversity and plot it next to the metric.
3. Swap the D3 multimodal objective and predict which operator needs retuning.